# Computação Gráfica – Projeto 2
Construção de um cenário 3D com ambiente interno e externo utilizando modelos `.obj`, texturas e câmera móvel.


A cena é composta por uma cabana/chalé com lareira, poltrona, mesa com gramofone e cinzeiro, além de um gato. A área externa mostra uma paisagem de neve com uma árvore e uma fogueira com dois troncos como assento.

Feito por:

* Nome completo --- NUSP
* Nicolas Carreiro Rodrigues --- 14600801

Lista dos controles:

> * **[W], [A], [S], [D]:** movimentação da câmera.
> * **Mouse:** direção da câmera.
> * **Scroll:** controle do campo de visão.
> * **[P]:** alternar visualização da malha poligonal.
> * **[ESC]:** fechar a janela.
> * **[F]** ascender a fogueira (escala).
> * **[G]** ligar o gramofone (rotação).
> * **[Setas]** controlar o gato (translação).

*Obs.: o notebook da aula 13 foi usado de base para este projeto.*

---
## Instalação e inicialização

### Primeiro, vamos importar as bibliotecas necessárias.

In [25]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [26]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 900
largura = 1600

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


---
## Shaders

### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [27]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

---
## Carregamento de modelos e texturas

### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [28]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

def load_obj_and_texture_fan(objFile, textureId, textureFile):
    modelo = load_model_from_file(objFile)

    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))

    for face in modelo['faces']:
        for i in range(1, len(face[0]) - 1):
            for vertice_id in [face[0][0], face[0][i], face[0][i + 1]]:
                vertices_list.append(modelo['vertices'][vertice_id - 1])
            for texture_id in [face[1][0], face[1][i], face[1][i + 1]]:
                textures_coord_list.append(modelo['texture'][texture_id - 1])

    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    load_texture_from_file(textureId, textureFile)

    return verticeInicial, verticeFinal - verticeInicial


def load_obj_and_texture_xy(objFile, textureId, textureFile):
    modelo = load_model_from_file(objFile)

    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))

    x_min = min([float(v[0]) for v in modelo['vertices']])
    x_max = max([float(v[0]) for v in modelo['vertices']])
    y_min = min([float(v[1]) for v in modelo['vertices']])
    y_max = max([float(v[1]) for v in modelo['vertices']])

    for face in modelo['faces']:
        for i in range(1, len(face[0]) - 1):
            for vertice_id in [face[0][0], face[0][i], face[0][i + 1]]:
                vertex = modelo['vertices'][vertice_id - 1]
                x = float(vertex[0])
                y = float(vertex[1])
                u = (x - x_min) / (x_max - x_min)
                v = (y - y_min) / (y_max - y_min)
                vertices_list.append(vertex)
                textures_coord_list.append((u, v))

    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    load_texture_from_file(textureId, textureFile)

    return verticeInicial, verticeFinal - verticeInicial


### Vamos carregar cada modelo e definir funções para desenhá-los

#### Modelos dos objetos

##### Ambiente interno

- **Gato**

In [29]:
verticeInicial_gato, quantosVertices_gato = load_obj_and_texture(
    'objetos/interno/gato/gato.obj',
    ['objetos/interno/gato/gato.jpg'] 
)

def desenha_gato(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_gato, quantosVertices_gato)

Processando modelo objetos/interno/gato/gato.obj. Vertice inicial: 0
Processando modelo objetos/interno/gato/gato.obj. Vertice final: 317592
0


- **Gramofone**

In [30]:
textureId_gramofone = glGenTextures(1)
verticeInicial_gramofone, quantosVertices_gramofone = load_obj_and_texture_fan(
    'objetos/interno/gramofone/gramofone.obj',
    textureId_gramofone,
    'objetos/interno/gramofone/gramofone.tif'
)

def desenha_gramofone(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_gramofone, quantosVertices_gramofone)

# CD ---------------------------------------------------------------------------------------

textureId_cd = glGenTextures(1)
verticeInicial_cd, quantosVertices_cd = load_obj_and_texture_fan(
    'objetos/interno/gramofone/cd.obj',
    textureId_cd,
    'objetos/interno/gramofone/cd.png'
)

def desenha_cd(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_cd, quantosVertices_cd)

Processando modelo objetos/interno/gramofone/gramofone.obj. Vertice inicial: 317592
Processando modelo objetos/interno/gramofone/gramofone.obj. Vertice final: 513849
1
Processando modelo objetos/interno/gramofone/cd.obj. Vertice inicial: 513849
Processando modelo objetos/interno/gramofone/cd.obj. Vertice final: 514617
2


- **Mesa**

In [31]:
textureId_mesa = glGenTextures(1)
verticeInicial_mesa, quantosVertices_mesa = load_obj_and_texture_fan(
    'objetos/interno/mesa/mesa.obj',
    textureId_mesa,
    'objetos/interno/mesa/mesa.png'
)

def desenha_mesa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_mesa, quantosVertices_mesa)

Processando modelo objetos/interno/mesa/mesa.obj. Vertice inicial: 514617
Processando modelo objetos/interno/mesa/mesa.obj. Vertice final: 624705
3


- **Cinzeiro**

In [32]:
textureId_cinzeiro = glGenTextures(1)
verticeInicial_cinzeiro, quantosVertices_cinzeiro = load_obj_and_texture_fan(
    'objetos/interno/cinzeiro/cinzeiro.obj',
    textureId_cinzeiro,
    'objetos/interno/cinzeiro/cinzeiro.png'
)

def desenha_cinzeiro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_cinzeiro, quantosVertices_cinzeiro)

Processando modelo objetos/interno/cinzeiro/cinzeiro.obj. Vertice inicial: 624705
Processando modelo objetos/interno/cinzeiro/cinzeiro.obj. Vertice final: 668745
4


- **Lareira**

In [33]:
textureId_lareira_1 = glGenTextures(1)
textureId_lareira_2 = glGenTextures(1)

modelo_lareira = load_model_from_file('objetos/interno/lareira/lareira.obj')

verticeInicial_lareira_1 = len(vertices_list)
for face in modelo_lareira['faces']:
    if face[2] == 'lambert2SG':
        for i in range(1, len(face[0]) - 1):
            for vertice_id in [face[0][0], face[0][i], face[0][i + 1]]:
                vertices_list.append(modelo_lareira['vertices'][vertice_id - 1])
            for texture_id in [face[1][0], face[1][i], face[1][i + 1]]:
                textures_coord_list.append(modelo_lareira['texture'][texture_id - 1])
quantosVertices_lareira_1 = len(vertices_list) - verticeInicial_lareira_1

verticeInicial_lareira_2 = len(vertices_list)
for face in modelo_lareira['faces']:
    if face[2] == 'lambert3SG':
        for i in range(1, len(face[0]) - 1):
            for vertice_id in [face[0][0], face[0][i], face[0][i + 1]]:
                vertices_list.append(modelo_lareira['vertices'][vertice_id - 1])
            for texture_id in [face[1][0], face[1][i], face[1][i + 1]]:
                textures_coord_list.append(modelo_lareira['texture'][texture_id - 1])
quantosVertices_lareira_2 = len(vertices_list) - verticeInicial_lareira_2

load_texture_from_file(textureId_lareira_1, 'objetos/interno/lareira/lareira1.png')
load_texture_from_file(textureId_lareira_2, 'objetos/interno/lareira/lareira2.png')

def desenha_lareira(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId_1, textureId_2):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId_1)
    glDrawArrays(GL_TRIANGLES, verticeInicial_lareira_1, quantosVertices_lareira_1)

    glBindTexture(GL_TEXTURE_2D, textureId_2)
    glDrawArrays(GL_TRIANGLES, verticeInicial_lareira_2, quantosVertices_lareira_2)

5
6


- **Poltrona**

In [34]:
textureId_poltrona = glGenTextures(1)
verticeInicial_poltrona, quantosVertices_poltrona = load_obj_and_texture_fan(
    'objetos/interno/poltrona/poltrona.obj',
    textureId_poltrona,
    'objetos/interno/poltrona/poltrona.png'
)

def desenha_poltrona(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_poltrona, quantosVertices_poltrona)

Processando modelo objetos/interno/poltrona/poltrona.obj. Vertice inicial: 670167
Processando modelo objetos/interno/poltrona/poltrona.obj. Vertice final: 672879
7


##### Ambiente externo

- **Fogueira**

In [35]:
# Fogo ---------------------------------------------------------------------------------------

textureId_fogo = glGenTextures(1)
verticeInicial_fogo, quantosVertices_fogo = load_obj_and_texture_xy(
    'objetos/externo/fogueira/fogo.obj',
    textureId_fogo,
    'objetos/externo/fogueira/fogo.jpg'
)

def desenha_fogo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_fogo, quantosVertices_fogo)


# Madeiras ---------------------------------------------------------------------------------------

textureId_madeiras = glGenTextures(1)
verticeInicial_madeiras, quantosVertices_madeiras = load_obj_and_texture_fan(
    'objetos/externo/fogueira/madeiras.obj',
    textureId_madeiras,
    'objetos/externo/fogueira/madeira.jpg'
)

def desenha_madeiras(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_madeiras, quantosVertices_madeiras)

Processando modelo objetos/externo/fogueira/fogo.obj. Vertice inicial: 672879
Processando modelo objetos/externo/fogueira/fogo.obj. Vertice final: 858285
8
Processando modelo objetos/externo/fogueira/madeiras.obj. Vertice inicial: 858285
Processando modelo objetos/externo/fogueira/madeiras.obj. Vertice final: 859149
9


- **Tronco**


In [36]:
textureId_tronco = glGenTextures(1)
verticeInicial_tronco, quantosVertices_tronco = load_obj_and_texture_fan(
    'objetos/externo/tronco/tronco.obj',
    textureId_tronco,
    'objetos/externo/tronco/tronco.jpg'
)

def desenha_tronco(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_tronco, quantosVertices_tronco)

Processando modelo objetos/externo/tronco/tronco.obj. Vertice inicial: 859149
Processando modelo objetos/externo/tronco/tronco.obj. Vertice final: 1020429
10


- **Tronco**


In [37]:
textureId_arvore = glGenTextures(1)
verticeInicial_arvore, quantosVertices_arvore = load_obj_and_texture_fan(
    'objetos/externo/arvore/arvore.obj',
    textureId_arvore,
    'objetos/externo/arvore/arvore.png'
)

def desenha_arvore(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_arvore, quantosVertices_arvore)

Processando modelo objetos/externo/arvore/arvore.obj. Vertice inicial: 1020429
Processando modelo objetos/externo/arvore/arvore.obj. Vertice final: 1135665
11


#### Modelos do ambiente
- Skybox
- Chão externo
- Chão interno

In [38]:
# Skybox ---------------------------------------------------------------------------------------

# Os shaders específicos da skybox servem para evitar as "costuras" nas divisões das faces do cubo.
skyboxVertexShaderSource = """
#version 330 core

attribute vec3 position;
varying vec3 out_texture;

uniform mat4 model;
uniform mat4 view;
uniform mat4 projection;

void main(){
    out_texture = position;
    vec4 pos = projection * view * model * vec4(position, 1.0);
    gl_Position = pos.xyww;
}
"""

skyboxFragmentShaderSource = """
#version 330 core

varying vec3 out_texture;
uniform samplerCube skybox;

void main(){
    gl_FragColor = texture(skybox, out_texture);
}
"""

skyboxVertexShader = glCreateShader(GL_VERTEX_SHADER)
glShaderSource(skyboxVertexShader, skyboxVertexShaderSource)
glCompileShader(skyboxVertexShader)

skyboxFragmentShader = glCreateShader(GL_FRAGMENT_SHADER)
glShaderSource(skyboxFragmentShader, skyboxFragmentShaderSource)
glCompileShader(skyboxFragmentShader)

skyboxProgram = glCreateProgram()
glAttachShader(skyboxProgram, skyboxVertexShader)
glAttachShader(skyboxProgram, skyboxFragmentShader)
glLinkProgram(skyboxProgram)

glDeleteShader(skyboxVertexShader)
glDeleteShader(skyboxFragmentShader)

glUseProgram(skyboxProgram)
loc_skybox = glGetUniformLocation(skyboxProgram, "skybox")
glUniform1i(loc_skybox, 0)
ourShader.use()

verticeInicial_skybox = len(vertices_list)

skybox_vertices = [
    (-1,  1, -1), (-1, -1, -1), (1, -1, -1),
    (1, -1, -1), (1,  1, -1), (-1,  1, -1),

    (-1, -1,  1), (-1, -1, -1), (-1,  1, -1),
    (-1,  1, -1), (-1,  1,  1), (-1, -1,  1),

    (1, -1, -1), (1, -1,  1), (1,  1,  1),
    (1,  1,  1), (1,  1, -1), (1, -1, -1),

    (-1, -1,  1), (-1,  1,  1), (1,  1,  1),
    (1,  1,  1), (1, -1,  1), (-1, -1,  1),

    (-1,  1, -1), (1,  1, -1), (1,  1,  1),
    (1,  1,  1), (-1,  1,  1), (-1,  1, -1),

    (-1, -1, -1), (-1, -1,  1), (1, -1, -1),
    (1, -1, -1), (-1, -1,  1), (1, -1,  1)
]

for vertex in skybox_vertices:
    vertices_list.append(vertex)
    textures_coord_list.append((0, 0))

quantosVertices_skybox = len(vertices_list) - verticeInicial_skybox

skybox_faces = [
    'objetos/ambiente/Skybox/posx.jpg',
    'objetos/ambiente/Skybox/negx.jpg',
    'objetos/ambiente/Skybox/posy.jpg',
    'objetos/ambiente/Skybox/negy.jpg',
    'objetos/ambiente/Skybox/posz.jpg',
    'objetos/ambiente/Skybox/negz.jpg'
]

textureId_skybox = glGenTextures(1)
glBindTexture(GL_TEXTURE_CUBE_MAP, textureId_skybox)

for i in range(len(skybox_faces)):
    img = Image.open(skybox_faces[i])
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, 1)
    glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

def desenha_skybox(t_x, t_y, t_z, textureId):

    glDepthFunc(GL_LEQUAL)
    glUseProgram(skyboxProgram)

    mat_model = model(0, 0, 0, 0, t_x, t_y, t_z, 50, 50, 50)
    loc_model = glGetUniformLocation(skyboxProgram, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    mat_view = view()
    loc_view = glGetUniformLocation(skyboxProgram, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(skyboxProgram, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices_skybox = glGetAttribLocation(skyboxProgram, "position")
    glEnableVertexAttribArray(loc_vertices_skybox)
    glVertexAttribPointer(loc_vertices_skybox, 3, GL_FLOAT, False, stride, offset)

    glBindTexture(GL_TEXTURE_CUBE_MAP, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_skybox, quantosVertices_skybox)

    glDepthFunc(GL_LESS)
    ourShader.use()


# Chao externo ---------------------------------------------------------------------------------------

verticeInicial_chao_ext = len(vertices_list)
textureId_chao_ext = glGenTextures(1)

vertices_chao_ext = [
    (-300, -0.5, -300), (300, -0.5, -300), (300, -0.5, 300),
    (-300, -0.5, -300), (300, -0.5, 300), (-300, -0.5, 300)
]

textures_chao_ext = [
    (0, 0), (60, 0), (60, 60),
    (0, 0), (60, 60), (0, 60)
]

for vertex in vertices_chao_ext:
    vertices_list.append(vertex)

for texture_coord in textures_chao_ext:
    textures_coord_list.append(texture_coord)

load_texture_from_file(textureId_chao_ext, 'objetos/ambiente/Chao_ext/chao_neve.jpg')

quantosVertices_chao_ext = len(vertices_list) - verticeInicial_chao_ext

def desenha_chao_ext(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_chao_ext, quantosVertices_chao_ext)


# Cabana ---------------------------------------------------------------------------------------

textureId_cabana = glGenTextures(1)
verticeInicial_cabana, quantosVertices_cabana = load_obj_and_texture_fan(
    'objetos/ambiente/Cabana/Snow covered CottageOBJ.obj',
    textureId_cabana,
    'objetos/ambiente/Cabana/Cottage Texture.jpg'
)

def desenha_cabana(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_cabana, quantosVertices_cabana)


# Chao interno ---------------------------------------------------------------------------------------

verticeInicial_chao_int = len(vertices_list)
textureId_chao_int = glGenTextures(1)

vertices_chao_int = [
    (-28, -2.35, -58), (28, -2.35, -58), (28, -2.35, 38),
    (-28, -2.35, -58), (28, -2.35, 38), (-28, -2.35, 38)
]

textures_chao_int = [
    (0, 0), (6, 0), (6, 10),
    (0, 0), (6, 10), (0, 10)
]

for vertex in vertices_chao_int:
    vertices_list.append(vertex)

for texture_coord in textures_chao_int:
    textures_coord_list.append(texture_coord)

load_texture_from_file(textureId_chao_int, 'objetos/ambiente/Chao_int/chao_madeira.jpg')

quantosVertices_chao_int = len(vertices_list) - verticeInicial_chao_int

def desenha_chao_int(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_chao_int, quantosVertices_chao_int)




13
Processando modelo objetos/ambiente/Cabana/Snow covered CottageOBJ.obj. Vertice inicial: 1135707
Processando modelo objetos/ambiente/Cabana/Snow covered CottageOBJ.obj. Vertice final: 1137489
14
15


---
## Envio ao buffer e alocação na GPU

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [39]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [40]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [41]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


---
## Eventos de teclado e câmera

### Eventos de teclado

**Posição da câmera:**
* Teclas A, S, D e W para movimentação no espaço tridimensional
* Posição do mouse para "direcionar" a câmera

**Demais eventos:**
* **[P]:** alternar visualização da malha poligonal.
* **[ESC]:** fechar a janela.
* **[F]** ascender a fogueira (escala).
* **[G]** ligar o gramofone (rotação).
* **[Setas]** controlar o gato (translação).

In [42]:
'''
Variáveis globais das transformações por eventos de teclado
'''

#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

# camera
cameraPos   = glm.vec3(-0.279745, 0.800512, 3.0662)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

# limites invisiveis da cena
limite_camera_x_min = -20.0
limite_camera_x_max = 20.0
limite_camera_y_min = 0.0
limite_camera_y_max = 10.0
limite_camera_z_min = -20.0
limite_camera_z_max = 20.0

# ciclo de escala do fogo
fogo_ativo = False
fogo_2_ativo = False
escala_fogo_min = 0.005
escala_fogo_max = 0.2
escala_fogo_1 = escala_fogo_min
escala_fogo_2 = escala_fogo_min
velocidade_escala_fogo = 0.12

# rotacao do CD
cd_girando = False
angulo_cd = 0.0
velocidade_cd = 180.0

# variável temporária usada para mexer objetos no cenário
# adicionar na chamada das funções de desenho dos objetos que quiser mover no lugar das coordenadas 
movedor_x = 0.0
movedor_z = 0.0
passo_movedor = 0.1




In [43]:
'''
Definição dos eventos de teclado
'''

def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode
    global fogo_ativo, fogo_2_ativo, escala_fogo_1, escala_fogo_2
    global cd_girando
    global movedor_x, movedor_z

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    cameraPos.x = max(limite_camera_x_min, min(cameraPos.x, limite_camera_x_max))
    cameraPos.y = max(limite_camera_y_min, min(cameraPos.y, limite_camera_y_max))
    cameraPos.z = max(limite_camera_z_min, min(cameraPos.z, limite_camera_z_max))

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if key == glfw.KEY_F and action == glfw.PRESS:
        fogo_ativo = not fogo_ativo
        fogo_2_ativo = False
        escala_fogo_1 = escala_fogo_min
        escala_fogo_2 = escala_fogo_min

    if key == glfw.KEY_G and action == glfw.PRESS:
        cd_girando = not cd_girando

    if action == glfw.PRESS or action == glfw.REPEAT:
        if key == glfw.KEY_RIGHT:
            movedor_x -= passo_movedor
        if key == glfw.KEY_LEFT:
            movedor_x += passo_movedor
        if key == glfw.KEY_DOWN:
            movedor_z -= passo_movedor
        if key == glfw.KEY_UP:
            movedor_z += passo_movedor
        if key in [glfw.KEY_LEFT, glfw.KEY_RIGHT, glfw.KEY_UP, glfw.KEY_DOWN]:
            print('movedor x:', round(movedor_x, 2), 'z:', round(movedor_z, 2))
        

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    

    

glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

---
## Matrizes de transformação

### Matrizes Model, View e Projection

In [44]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 500.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

---
## Janela final

### Nesse momento, nós exibimos a janela!


In [45]:
glfw.maximize_window(window)
glfw.show_window(window)

### Loop principal da janela.

In [46]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    if cd_girando:
        angulo_cd += velocidade_cd * deltaTime

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

    
    desenha_skybox(cameraPos.x, cameraPos.y, cameraPos.z, textureId_skybox)

    '''
    Guia das funções desenha_objeto:
        1. ângulo de rotação (em graus)
        2. eixos x, y, z ('1' aplica o ângulo de rotação integralmente, '0' não aplica rotação)
        3. deslocamento em x, y, z 
        4. escala em x, y, z
        5. id da textura a ser aplicada
    '''
    desenha_chao_ext(0, 0, 0, 0, 0, 0, 0, 1, 1, 1, textureId_chao_ext)
    desenha_cabana(225, 0, 1, 0, -7, -0.5, -12, 0.08, 0.08, 0.08, textureId_cabana)
    desenha_chao_int(225, 0, 1, 0, -7.5, 0.0001, -12.5, 0.079, 0.08, 0.075, textureId_chao_int)
    desenha_gramofone(0, 0, 1, 0, -6.75, 0.42, -13.55, 0.012, 0.012, 0.012, textureId_gramofone)
    desenha_cd(angulo_cd, 0, 1, 0, -6.75 , 0.698, -13.55 , 0.2, 0.2, 0.2, textureId_cd)
    desenha_mesa(0, 0, 1, 0, -6.75, -0.15, -13.35, 0.4, 0.4, 0.4, textureId_mesa)
    desenha_cinzeiro(225, 0, 1, 0, -6.6, 0.42, -13.1, 0.015, 0.015, 0.015, textureId_cinzeiro)
    desenha_lareira(-225, 0, 1, 0, -7.5, -0.15, -9.9, 0.01, 0.01, 0.01, textureId_lareira_1, textureId_lareira_2)
    desenha_poltrona(315, 0, 1, 0, -5.7, -0.25, -12.5, 0.008, 0.008, 0.008, textureId_poltrona)
    desenha_gato(-90, 1, 0, 0, -7.5+movedor_x, -0.2, -12+movedor_z, 0.015, 0.015, 0.015, 0)
    desenha_tronco(-90, 1, 0, 0, -0.2, -0.4, 2, 0.12, 0.045, 0.045, textureId_tronco)
    desenha_tronco(-90, 1, 0, 0, -0.2, -0.4, -2, 0.12, 0.045, 0.045, textureId_tronco)
    desenha_arvore(0, 0, 0, 0, 0, -1, -20, 7, 7, 7, textureId_arvore)
    desenha_madeiras(90, 0, 1, 0, -0.05, -0.1, -0.05, 0.2, 0.2, 0.2, textureId_madeiras)

    '''
    Esse ciclo faz com que 2 chamas fiquem crescendo, sumindo e reaparecendo
    repetidamente, dando a impressão de um fogo "vivo".
    '''
    fogo_x = 0
    fogo_y = -0.3
    fogo_z = 0

    if fogo_ativo:
        escala_fogo_1 += velocidade_escala_fogo * deltaTime
    if escala_fogo_1 >= escala_fogo_max / 2:
        fogo_2_ativo = True
    if escala_fogo_1 >= escala_fogo_max:
        escala_fogo_1 = escala_fogo_min

    if fogo_2_ativo:
        escala_fogo_2 += velocidade_escala_fogo * deltaTime
        if escala_fogo_2 >= escala_fogo_max:
            escala_fogo_2 = escala_fogo_min

    if fogo_ativo:
        desenha_fogo(0, 0, 0, 0, fogo_x, fogo_y, fogo_z, escala_fogo_1, escala_fogo_1, escala_fogo_1, textureId_fogo)
        if fogo_2_ativo:
            desenha_fogo(0, 0, 0, 0, fogo_x-0.1, fogo_y, fogo_z, escala_fogo_2, escala_fogo_2, escala_fogo_2, textureId_fogo)
    
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)

glfw.terminate()